# Area C — BERTurk fine-tuning (SWE015 UC4)

Fine-tunes `dbmdz/bert-base-turkish-cased` on **OffensEval-TR 2020** for offensive-language classification.
Run this in **Google Colab** with a GPU runtime (Runtime → Change runtime type → GPU).

Outputs: macro-F1 on the shared test set, per-message CPU latency, model size, and a saved model for the Area D (quantization) step.

In [1]:
!pip -q install "transformers>=4.40" "datasets>=2.18" scikit-learn

In [2]:
import re, time, os, numpy as np, torch
import torch.nn as nn
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from sklearn.metrics import f1_score, classification_report

MODEL = "dbmdz/bert-base-turkish-cased"
LABEL_NAMES = ["NOT", "OFF"]   # 0 = clean, 1 = offensive
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


## Load the shared dataset (same split every model uses — R3)

In [4]:
ds = load_dataset("stefan-it/offenseval2020_tr")
tok = AutoTokenizer.from_pretrained(MODEL)

# this version stores ONE "text" column in fastText format: "__label__OFF <message>"
LABEL_RE = re.compile(r"^__label__(NOT|OFF)\s+")
def split_label(raw):
    m = LABEL_RE.match(raw)
    label = LABEL_NAMES.index(m.group(1)) if m else 0
    text = LABEL_RE.sub("", raw, count=1)
    return text, label

def encode(ex):
    text, label = split_label(ex["text"])
    text = re.sub(r"@USER|URL|http\S+", " ", text)
    enc = tok(text, truncation=True, max_length=128)
    enc["labels"] = label
    return enc

cols = ds["train"].column_names
ds_enc = ds.map(encode, remove_columns=cols)
collator = DataCollatorWithPadding(tok)

train_labels = [split_label(t)[1] for t in ds["train"]["text"]]
counts = np.bincount(train_labels, minlength=2)
class_weights = torch.tensor(counts.sum() / (2 * counts), dtype=torch.float32)
print("class counts:", counts, "| weights:", class_weights.tolist())

Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1756 [00:00<?, ? examples/s]

Map:   0%|          | 0/3528 [00:00<?, ? examples/s]

class counts: [24204  5796] | weights: [0.6197322607040405, 2.587991714477539]


## Fine-tune (weighted loss for class imbalance)

In [7]:
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, **kw):
        super().__init__(**kw)
        self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = nn.CrossEntropyLoss(weight=self.class_weights.to(model.device))(
            outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)

args = TrainingArguments(
    output_dir="berturk-ckpt",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    logging_steps=100,
    save_strategy="no",
    report_to="none",
)

trainer = WeightedTrainer(
    model=model, args=args,
    train_dataset=ds_enc["train"],
    data_collator=collator, processing_class=tok,
    class_weights=class_weights,
)
trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
100,0.672577
200,0.536347
300,0.427146
400,0.460615
500,0.449979
600,0.447863
700,0.415478
800,0.402925
900,0.422440
1000,0.436415


TrainOutput(global_step=5625, training_loss=0.3082752849155002, metrics={'train_runtime': 1018.2982, 'train_samples_per_second': 88.383, 'train_steps_per_second': 5.524, 'total_flos': 2936039822321280.0, 'train_loss': 0.3082752849155002, 'epoch': 3.0})

## Evaluate on the shared test set (macro-F1)

In [8]:
pred = trainer.predict(ds_enc["test"])
y_pred = pred.predictions.argmax(-1)
y_true = pred.label_ids

macro_f1 = f1_score(y_true, y_pred, average="macro")
off_f1 = f1_score(y_true, y_pred, pos_label=1)
print(f"Macro-F1:     {macro_f1:.4f}")
print(f"Offensive F1: {off_f1:.4f}\n")
print(classification_report(y_true, y_pred, target_names=LABEL_NAMES))

Macro-F1:     0.8120
Offensive F1: 0.6989

              precision    recall  f1-score   support

         NOT       0.92      0.93      0.93      2812
         OFF       0.71      0.69      0.70       716

    accuracy                           0.88      3528
   macro avg       0.82      0.81      0.81      3528
weighted avg       0.88      0.88      0.88      3528



## Per-message latency on CPU + model size (R4)
The scenario is a CPU-only server, so latency is measured on CPU.

In [9]:
cpu_model = model.to("cpu").eval()
samples = [re.sub(r"@USER|URL|http\S+", " ", split_label(t)[0]) for t in ds["test"]["text"][:50]]
pool = (samples * 4)[:200]

with torch.no_grad():
    t0 = time.perf_counter()
    for s in pool:
        enc = tok(s, truncation=True, max_length=128, return_tensors="pt")
        cpu_model(**enc)
    latency_ms = (time.perf_counter() - t0) / len(pool) * 1000
print(f"Latency: {latency_ms:.1f} ms/msg (CPU — state the Colab CPU in the report)")

Latency: 124.4 ms/msg (CPU — state the Colab CPU in the report)


In [10]:
save_dir = "berturk-offenseval"
model.save_pretrained(save_dir)
tok.save_pretrained(save_dir)
size_mb = sum(os.path.getsize(os.path.join(save_dir, f))
              for f in os.listdir(save_dir)) / 1e6
print(f"Model size: {size_mb:.1f} MB (FP32 — Area D will quantize this)")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model size: 443.3 MB (FP32 — Area D will quantize this)


## Save the model for the Area D (quantization) step
Download the zip, or mount Drive and copy it there. **Do not commit it to the repo** — link to Hugging Face Hub or Drive instead (R7).

In [11]:
!zip -q -r berturk-offenseval.zip berturk-offenseval
from google.colab import files
files.download("berturk-offenseval.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
from huggingface_hub import login
login()   # çıkan kutuya Write token'ını yapıştır

In [13]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
m = AutoModelForSequenceClassification.from_pretrained("berturk-offenseval")
t = AutoTokenizer.from_pretrained("berturk-offenseval")
repo = "tia88na/berturk-offenseval-tr"
m.push_to_hub(repo)
t.push_to_hub(repo)
print("Model URL:", "https://huggingface.co/" + repo)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...uo6suj1/model.safetensors:   0%|          |  556kB /  442MB            

Validating                    :           |   0%            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Model URL: https://huggingface.co/tia88na/berturk-offenseval-tr
